🧩 Bloque 1 — Importación de librerías

Importa todas las librerías necesarias para la automatización del proceso:
- pandas y numpy para el manejo y procesamiento de datos.
- sqlalchemy para la conexión e inserción de información en SQL Server.
- datetime, pathlib, os, re y unicodedata para manipular fechas, rutas y normalizar textos.
- decimal para controlar redondeos y precisión numérica en cálculos financieros.

In [ ]:
import pandas as pd
from datetime import datetime
from sqlalchemy import text
import sys
import importlib
import json
import os
from os import path
from pathlib import Path

# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks/banigualdad/
# queremos:  .../traspaso-innominados/functions
#            .../traspaso-innominados/config
PROJECT_ROOT = Path(os.environ["PROJECT_ROOT"]).resolve()
FUNCTIONS_DIR = PROJECT_ROOT / "functions"
CONFIG_DIR = PROJECT_ROOT / "config"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())
print("CONFIG_DIR:",    CONFIG_DIR,    "exists:", CONFIG_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
module_name = "functions"
try:
    if module_name in sys.modules:
        fun = importlib.reload(sys.modules[module_name])
    else:
        fun = importlib.import_module(module_name)
    print("Módulo importado desde:", getattr(fun, "__file__", "<sin __file__>"))
except Exception as e:
    raise ImportError(
        f"No se pudo importar el módulo '{module_name}'.\n"
        f"Revisa que exista en: {FUNCTIONS_DIR}\n"
        f"Detalle: {e}"
    )

# --- Cargar config JSON ---
config_file = path.join(CONFIG_DIR, "losheroes", "config_losheroes.json")
try:
    with open(config_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("Config cargada desde:", config_file)
    print("Claves disponibles:", list(data.keys()))
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración:\n{config_file}\n"
        f"¿Existe CONFIG_DIR?: {CONFIG_DIR.exists()}"
    )
except json.JSONDecodeError as e:
    raise ValueError(f"Error: JSON inválido en {config_file}: {e}")
except Exception as e:
    raise RuntimeError(f"Error inesperado leyendo {config_file}: {e}")


PROJECT_ROOT: C:\Users\dasanchez\OneDrive - Seguros Suramericana, S.A\Documentos\data_analysis_projects\traspaso-innominados
FUNCTIONS_DIR: C:\Users\dasanchez\OneDrive - Seguros Suramericana, S.A\Documentos\data_analysis_projects\traspaso-innominados\functions exists: True
CONFIG_DIR: C:\Users\dasanchez\OneDrive - Seguros Suramericana, S.A\Documentos\data_analysis_projects\traspaso-innominados\config exists: True
Módulo importado desde: C:\Users\dasanchez\OneDrive - Seguros Suramericana, S.A\Documentos\data_analysis_projects\traspaso-innominados\functions\functions.py
Config cargada desde: C:\Users\dasanchez\OneDrive - Seguros Suramericana, S.A\Documentos\data_analysis_projects\traspaso-innominados\config\losheroes\config_losheroes.json
Claves disponibles: ['tablas_remotas', 'server_config', 'periodo']


🗄️ Bloque 2 — Conexión a la base de datos

Configura la conexión con la base de datos SQL Server utilizando SQLAlchemy.

Define los parámetros del servidor (server, database, schema, table) y construye la cadena de conexión (connection_string), la cual se usará al momento de insertar los datos procesados en la tabla Prueba_LosHeroes.

In [3]:
# --- Parámetros de conexión ---
server   = data["server_config"]["server"]
database = data["server_config"]["database"]
schema   = data["server_config"]["schema"]
table    = data["tablas_remotas"]["tabla_principal"]

driver = "ODBC Driver 17 for SQL Server"

# ODBC connection string (pyodbc) → para query_to_df / df_to_db / exec_sql
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# SQLAlchemy engine
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise"
)


📂 Bloque 3 — Configuración inicial del proceso

Define las variables globales del proceso:
- Ruta de la carpeta que contiene los archivos Excel.
- Índice de la hoja a procesar y el rango de columnas a leer.
- Extensiones válidas de archivo (.xlsx).
- Expresiones regulares para detectar el período (YYYYMM) y el número de póliza en los nombres de archivo o carpetas.

In [ ]:
# Cambiar el periodo en config/losheroes/config_losheroes.json
PERIODO      = data.get("periodo")
RUTA_CARPETA = (PROJECT_ROOT / "data" / "los_heroes" / PERIODO).resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_CARPETA:", RUTA_CARPETA, "exists:", RUTA_CARPETA.exists())

SHEET_INDEX = 1
USECOLS     = "A:M"  # Solo columnas A a M
EXTS        = (".xlsx",)


PROJECT_ROOT: C:\Users\dasanchez\OneDrive - Seguros Suramericana, S.A\Documentos\data_analysis_projects\traspaso-innominados
RUTA_CARPETA: C:\Users\dasanchez\OneDrive - Seguros Suramericana, S.A\Documentos\data_analysis_projects\traspaso-innominados\data\los_heroes\202513 exists: False


🧰 Bloque 4 — Funciones auxiliares

Incluye las funciones de apoyo que se utilizan en todo el flujo:
- Extracción del período y la póliza desde el nombre del archivo o la carpeta.
- `coalesce_cols` para combinar columnas equivalentes de distintas fuentes.
- `fun.normalize_name` para normalizar nombres de columnas.

In [ ]:
import re
import unicodedata

RE_POLIZA_7 = re.compile(r"(\d{7})(?=\D?$)")
RE_PERIODO  = re.compile(r"(20\d{2})(0[1-9]|1[0-2])")


def extract_period_from_dir(dir_path):
    m = RE_PERIODO.search(dir_path.name)
    return int(m.group(1) + m.group(2)) if m else None


def extract_period_from_filename(name: str):
    m = RE_PERIODO.search(name)
    return int(m.group(1) + m.group(2)) if m else None


def extract_poliza_from_filename(name: str):
    base = __import__("pathlib").Path(name).stem
    m = RE_POLIZA_7.search(base)
    return m.group(1) if m else None


def coalesce_cols(df, names):
    import pandas as pd
    cols = [c for c in names if c in df.columns]
    if not cols:
        return pd.Series([None] * len(df), index=df.index)
    out = df[cols[0]].where(df[cols[0]].notna(), None)
    for c in cols[1:]:
        out = out.where(out.notna(), df[c])
    return out


🧰 Bloque 4 — Funciones auxiliares

Incluye las funciones de apoyo que se utilizan en todo el flujo:
- Extracción del período y la póliza desde el nombre del archivo o la carpeta.
- Normalización de nombres de columnas y eliminación de acentos.
- Función coalesce_cols que permite combinar valores de columnas equivalentes para mantener consistencia en los datos de diferentes fuentes.

📑 Bloque 5 — Lectura y consolidación de archivos

Lee todos los archivos Excel ubicados en la carpeta definida.

Para cada archivo:
- Se cargan solo las columnas relevantes.
- Se limpian los valores nulos o vacíos.
- Se normalizan los nombres de las columnas.
- Se agregan columnas adicionales: poliza, periodo y nombre_archivo.
    
Finalmente, se concatenan todos los DataFrames en uno solo (df_all) para trabajar de manera unificada.

In [ ]:
assert RUTA_CARPETA.is_dir(), f"No existe carpeta: {RUTA_CARPETA}"
periodo_dir = extract_period_from_dir(RUTA_CARPETA)

excels = sorted(
    [p for p in RUTA_CARPETA.iterdir() if p.is_file() and p.suffix.lower() in EXTS],
    key=lambda p: p.name.lower()
)
assert len(excels) > 0, f"No hay Excel en {RUTA_CARPETA}"
print("Excel encontrados:", [p.name for p in excels])

frames = []
for p in excels:
    df = pd.read_excel(p, sheet_name=SHEET_INDEX, usecols=USECOLS, dtype=str)
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].astype(str).str.strip().replace({"None": "", "nan": ""})
   
    df.columns = [fun.normalize_name(c) for c in df.columns]
    df["poliza"]  = extract_poliza_from_filename(p.name)                 # <-- minúscula
    df["periodo"] = periodo_dir if periodo_dir else extract_period_from_filename(p.name)  # <-- minúscula
    df["nombre_archivo"] = p.name 

    frames.append(df)

df_all = pd.concat(frames, ignore_index=True)
print("Columnas normalizadas en df_all:", list(df_all.columns))

fun.pretty_table(df_all)

Excel encontrados: ['2025.11 REC_SURA_CES_TRABN2020_22_112025 6343333.xlsx', '2025.11 REC_SURA_CES_TRABN2023_27_112025 7694100.xlsx', '2025.11 REC_SURA_CES_TRBN2023_27_1_112025 7833136.xlsx', '2025.11 REC_SURA_CES_TRBN2024_27_112025 8523694.xlsx']
Columnas normalizadas en df_all: ['rut', 'operacion', 'digito_verificador', 'tipo_de_cliente', 'prima', 'moneda', 'nombre_asegurado', 'prima_neta', 'plazo', 'tramo', 'mes_venta', 'monto_credito', 'monto_cuota', 'poliza', 'periodo', 'nombre_archivo', 'numero_de_operacion', 'prima_bruta']


,rut,operacion,digito_verificador,tipo_de_cliente,prima,moneda,nombre_asegurado,prima_neta,plazo,tramo,mes_venta,monto_credito,monto_cuota,poliza,periodo,nombre_archivo,numero_de_operacion,prima_bruta
0,13389383,105432758,0,1,177800000,PS,Solange Eliana Acevedo Ortiz,15183.603757472245,60,48-60,2021-08-01 00:00:00,3065478,51091.3,6343333,202511,2025.11 REC_SURA_CES_TRABN2020_22_112025 63433...,NaN,NaN
1,13391009,105641524,3,1,52680000,PS,Ramón Segundo Piñaleo Llaulén,4498.719043552519,60,48-60,2022-02-01 00:00:00,1816532,30275.533333333333,6343333,202511,2025.11 REC_SURA_CES_TRABN2020_22_112025 63433...,NaN,NaN
2,13393261,106048797,5,1,46810000,PS,Marcia Noelia Valenzuela Salgado,3997.4380871050384,48,37-48,2022-12-01 00:00:00,1614012,33625.25,6343333,202511,2025.11 REC_SURA_CES_TRABN2020_22_112025 63433...,NaN,NaN
3,13401139,105256714,4,1,27890000,PS,Yasna Del Pilar Sandoval Mardones,2381.725021349274,60,48-60,2021-01-01 00:00:00,961694,16028.233333333334,6343333,202511,2025.11 REC_SURA_CES_TRABN2020_22_112025 63433...,NaN,NaN
4,13420875,105663236,9,1,18610000,PS,Leyla Yessenia Bernarda Guerra Guerra,1589.2399658411614,60,48-60,2022-03-01 00:00:00,641759,10695.983333333334,6343333,202511,2025.11 REC_SURA_CES_TRABN2020_22_112025 63433...,NaN,NaN


🧱 Bloque 6 — Construcción del DataFrame canónico

A partir del DataFrame consolidado (df_all), se estandarizan las columnas hacia un formato único utilizando coalesce_cols, que combina sinónimos o variaciones de nombres entre diferentes archivos.

El resultado es el DataFrame df_can, con todas las columnas alineadas a la estructura esperada por la base de datos SQL.

In [ ]:

Numero_Operacion = coalesce_cols(df_all, {"numero_operacion", "operacion", "n_operacion", "num_operacion", "nro_operacion", "numero_de_operacion"})
RUT              = coalesce_cols(df_all, {"rut"})
DV_RUT           = coalesce_cols(df_all, {"dv_rut", "digito_verificador", "dv"})

Tipo_Cliente     = coalesce_cols(df_all, {"tipo_cliente", "tipo_de_cliente"})
PRIMA            = coalesce_cols(df_all, {"prima", "prima_bruta"})
tipo_Moneda      = coalesce_cols(df_all, {"tipo_moneda", "moneda", "tipo_de_moneda"})
Nombre_Asegurado = coalesce_cols(df_all, {"nombre_asegurado", "asegurado", "nombre_cliente", "cliente"})
Prima_Neta       = coalesce_cols(df_all, {"prima_neta"})
Plazo            = coalesce_cols(df_all, {"plazo"})
Tramo            = coalesce_cols(df_all, {"tramo"})
Mes_venta        = coalesce_cols(df_all, {"mes_venta", "fecha_venta"})
Monto_Credito    = coalesce_cols(df_all, {"monto_credito"})
Monto_Cuota      = coalesce_cols(df_all, {"monto_cuota"})
Poliza           = coalesce_cols(df_all, {"poliza"})
Mes_Referencia   = coalesce_cols(df_all, {"periodo", "mes_referencia"})

df_can = pd.DataFrame({
    "Mes_Referencia": Mes_Referencia,
    "Poliza": Poliza,
    "Numero_Operacion": Numero_Operacion,
    "RUT": RUT,
    "DV_RUT": DV_RUT,
    "Tipo_Cliente": Tipo_Cliente,
    "PRIMA": PRIMA,
    "tipo_Moneda": tipo_Moneda,
    "Nombre_Asegurado": Nombre_Asegurado,
    "Prima_Neta": Prima_Neta,
    "Plazo": Plazo,
    "Tramo": Tramo,
    "Mes_venta": Mes_venta,
    "Monto_Credito": Monto_Credito,
    "Monto_Cuota": Monto_Cuota,
})

fun.pretty_table(df_can)

,Mes_Referencia,Poliza,Numero_Operacion,RUT,DV_RUT,Tipo_Cliente,PRIMA,tipo_Moneda,Nombre_Asegurado,Prima_Neta,Plazo,Tramo,Mes_venta,Monto_Credito,Monto_Cuota
0,202511,6343333,105432758,13389383,0,1,177800000,PS,Solange Eliana Acevedo Ortiz,15183.603757472245,60,48-60,2021-08-01 00:00:00,3065478,51091.3
1,202511,6343333,105641524,13391009,3,1,52680000,PS,Ramón Segundo Piñaleo Llaulén,4498.719043552519,60,48-60,2022-02-01 00:00:00,1816532,30275.533333333333
2,202511,6343333,106048797,13393261,5,1,46810000,PS,Marcia Noelia Valenzuela Salgado,3997.4380871050384,48,37-48,2022-12-01 00:00:00,1614012,33625.25
3,202511,6343333,105256714,13401139,4,1,27890000,PS,Yasna Del Pilar Sandoval Mardones,2381.725021349274,60,48-60,2021-01-01 00:00:00,961694,16028.233333333334
4,202511,6343333,105663236,13420875,9,1,18610000,PS,Leyla Yessenia Bernarda Guerra Guerra,1589.2399658411614,60,48-60,2022-03-01 00:00:00,641759,10695.983333333334


⚙️ Bloque 7 — Limpieza y tipificación de datos

Transforma los datos a sus tipos correctos:
- Convierte campos numéricos y fechas al formato estándar.
- Limpia RUTs, dígitos verificadores y cadenas de texto.
- Redondea valores decimales y normaliza formatos de fecha (YYYY-MM-DD).

Este paso garantiza la compatibilidad de los tipos de datos con la estructura SQL.

In [ ]:
def to_int_or_none(x):
    if pd.isna(x) or str(x).strip()=="":
        return None
    try:
        return int(float(str(x)))
    except:
        return None


def to_dec_str(x, nd=2):
    if pd.isna(x) or str(x).strip()=="":
        return None
    try:
        v = float(str(x).replace(",", "."))
        return f"{v:.{nd}f}"
    except:
        return None


def to_date_yyyy_mm_dd(x):
    if pd.isna(x) or str(x).strip()=="":
        return None
    s = str(x).strip()
    for fmt in ("%Y-%m-%d","%d-%m-%Y","%d/%m/%Y","%Y/%m/%d"):
        try:
            return datetime.strptime(s, fmt).strftime("%Y-%m-%d")
        except:
            pass
    try:
        f = float(s)
        if 1 < f < 600000:
            return pd.to_datetime(f, unit="D", origin="1899-12-30", errors="coerce").strftime("%Y-%m-%d")
    except:
        pass
    try:
        return pd.to_datetime(s, errors="coerce").strftime("%Y-%m-%d")
    except:
        return None

df_can["Mes_Referencia"]   = pd.to_numeric(df_can["Mes_Referencia"], errors="coerce").astype("Int64")
df_can["Numero_Operacion"] = df_can["Numero_Operacion"].map(to_int_or_none).astype("Int64")
df_can["Plazo"]            = df_can["Plazo"].map(to_int_or_none).astype("Int64")

df_can["RUT"]    = df_can["RUT"].map(lambda x: re.sub(r"\D", "", str(x)) if pd.notna(x) else None)
df_can["DV_RUT"] = df_can["DV_RUT"].map(lambda x: str(x).strip().upper() if pd.notna(x) else None)

for col in ["Tipo_Cliente", "tipo_Moneda", "Nombre_Asegurado", "Tramo", "Poliza"]:
    df_can[col] = df_can[col].astype("string").str.strip()

for col in ["PRIMA", "Prima_Neta", "Monto_Credito", "Monto_Cuota"]:
    df_can[col] = df_can[col].map(lambda x: to_dec_str(x, 2))

df_can["Mes_venta"] = df_can["Mes_venta"].map(to_date_yyyy_mm_dd)

fun.pretty_table(df_can)

,Mes_Referencia,Poliza,Numero_Operacion,RUT,DV_RUT,Tipo_Cliente,PRIMA,tipo_Moneda,Nombre_Asegurado,Prima_Neta,Plazo,Tramo,Mes_venta,Monto_Credito,Monto_Cuota
0,202511,6343333,105432758,13389383,0,1,177800000.00,PS,Solange Eliana Acevedo Ortiz,15183.60,60,48-60,2021-08-01,3065478.00,51091.30
1,202511,6343333,105641524,13391009,3,1,52680000.00,PS,Ramón Segundo Piñaleo Llaulén,4498.72,60,48-60,2022-02-01,1816532.00,30275.53
2,202511,6343333,106048797,13393261,5,1,46810000.00,PS,Marcia Noelia Valenzuela Salgado,3997.44,48,37-48,2022-12-01,1614012.00,33625.25
3,202511,6343333,105256714,13401139,4,1,27890000.00,PS,Yasna Del Pilar Sandoval Mardones,2381.73,60,48-60,2021-01-01,961694.00,16028.23
4,202511,6343333,105663236,13420875,9,1,18610000.00,PS,Leyla Yessenia Bernarda Guerra Guerra,1589.24,60,48-60,2022-03-01,641759.00,10695.98


🧾 Bloque 8 — Validación de RUTs

Verifica la validez de los RUTs contenidos en el DataFrame.

Se detectan registros con formatos sospechosos (por ejemplo, demasiados dígitos o caracteres no válidos) y se muestran los casos detectados para revisión manual antes de la carga a la base de datos.

In [16]:
def es_rut_valido(rut_str):
    """
    Verifica si el RUT parece válido por formato (no calcula DV real, solo estructura).
    Retorna True si parece un RUT correcto, False si no.
    """
    if pd.isna(rut_str): 
        return False
    s = str(rut_str).strip().replace(".", "").replace("-", "")
    # Un RUT válido tiene entre 7 y 8 dígitos (sin contar el DV)
    return s.isdigit() and 6 <= len(s) <= 8

# Detectar RUTs sospechosos (por ejemplo, con más de 8 dígitos)
mask_invalid_rut = ~df_can["RUT"].map(es_rut_valido)

# Mostrar ejemplos de registros sospechosos
invalid_rows = df_can.loc[mask_invalid_rut, ["RUT", "Numero_Operacion", "Nombre_Asegurado", "Poliza"]]
print(f"⚠️ Filas con RUT inválido o sospechoso: {len(invalid_rows)}")
display(invalid_rows.head(10))

⚠️ Filas con RUT inválido o sospechoso: 0


,RUT,Numero_Operacion,Nombre_Asegurado,Poliza


🧩 Bloque 9 — Preparación final del DataFrame

Selecciona y ordena las columnas en el formato final que debe coincidir exactamente con la estructura de la tabla.

Este DataFrame (df_sql) es el que se utilizará directamente para la inserción en SQL Server.

In [17]:
cols_sql = [
    "Mes_Referencia","Poliza","Numero_Operacion","RUT","DV_RUT",
    "Tipo_Cliente","PRIMA","tipo_Moneda","Nombre_Asegurado","Prima_Neta",
    "Plazo","Tramo","Mes_venta","Monto_Credito","Monto_Cuota"
]
df_sql = df_can[cols_sql].copy()

df_sql.head(3)

,Mes_Referencia,Poliza,Numero_Operacion,RUT,DV_RUT,Tipo_Cliente,PRIMA,tipo_Moneda,Nombre_Asegurado,Prima_Neta,Plazo,Tramo,Mes_venta,Monto_Credito,Monto_Cuota
0,202511,6343333,105432758,13389383,0,1,177800000.00,PS,Solange Eliana Acevedo Ortiz,15183.60,60,48-60,2021-08-01,3065478.00,51091.30
1,202511,6343333,105641524,13391009,3,1,52680000.00,PS,Ramón Segundo Piñaleo Llaulén,4498.72,60,48-60,2022-02-01,1816532.00,30275.53
2,202511,6343333,106048797,13393261,5,1,46810000.00,PS,Marcia Noelia Valenzuela Salgado,3997.44,48,37-48,2022-12-01,1614012.00,33625.25


🔍 Bloque 10 — Validación de período y control de duplicados

Valida que el campo Mes_Referencia sea único dentro del conjunto de datos.

Luego consulta la base de datos para confirmar que el período no haya sido cargado previamente.

Si ya existe, el proceso se detiene automáticamente para evitar duplicaciones en la tabla.

In [18]:
assert "Mes_Referencia" in df_sql.columns, "Falta la columna 'Mes_Referencia' en df_sql."

df_sql["Mes_Referencia"] = (
    df_sql["Mes_Referencia"]
    .astype("string")
    .str.strip()
)

vals = [
    v for v in df_sql["Mes_Referencia"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚨 No se encontraron valores válidos en 'Mes_Referencia'.")

counts = (
    df_sql["Mes_Referencia"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"📄 Se detectaron {len(vals)} Mes_Referencia distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

📄 Se detectaron 1 Mes_Referencia distintos en el df_sql:
   - 202511: 18517 filas


⬆️ Bloque 11 – Carga con reemplazo selectivo por archivo (ETL transaccional)

Este bloque implementa el core del proceso ETL:
- Se ejecuta una sola transacción.
- Para cada NOMBRE_ARCHIVO:
- Consulta si existe data previa en SQL.
- Si existe → elimina solo ese archivo (reemplazo controlado).
- Inserta solo las filas nuevas.
- Imprime un resumen:
    - Filas reemplazadas
    - Filas insertadas
    - Archivos nuevos vs antiguos

Es un flujo robusto tipo "upsert por archivo".

In [19]:
resumen = []

with engine.begin() as conn:
    qry_count = text(f"""
        SELECT COUNT(*) AS cnt
        FROM {schema}.{table}
        WHERE Mes_Referencia = :nombre
    """)

    qry_del = text(f"""
        DELETE FROM {schema}.{table}
        WHERE Mes_Referencia = :nombre
    """)

    for Mes_Referencia in vals: 
        df_sub = df_sql[df_sql["Mes_Referencia"] == Mes_Referencia]

        if df_sub.empty:
            print(f"⚠️ No se encontraron filas en df_sql para Mes_Referencia = '{Mes_Referencia}'. Se omite.")
            continue

        existing_count = conn.execute(qry_count, {"nombre": Mes_Referencia}).scalar() or 0

        if existing_count > 0:
            print(f"♻️ Se encontró data previa para Mes_Referencia = '{Mes_Referencia}' "
                  f"({existing_count} filas en {schema}.{table}).")
            print("🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...")

            deleted = conn.execute(qry_del, {"nombre": Mes_Referencia}).rowcount
            print(f"✅ Filas eliminadas en destino para '{Mes_Referencia}': {deleted}")
            accion = "reemplazo"
        else:
            print(f"🆕 No hay data previa para Mes_Referencia = '{Mes_Referencia}'. "
                  "Se insertará como archivo nuevo.")
            accion = "inserción nueva"

        df_sub.to_sql(
            name=table,
            con=conn,       
            schema=schema,
            if_exists='append',
            index=False
        )

        filas_sub = len(df_sub)
        print(f"📥 Insertadas {filas_sub} filas para Mes_Referencia = '{Mes_Referencia}'.")
        resumen.append((Mes_Referencia, filas_sub, existing_count, accion))


print("\n📊 Resumen de carga por Mes_Referencia:")
for Mes_Referencia, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {Mes_Referencia}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {Mes_Referencia}: {filas_sub} filas cargadas (archivo nuevo).")


🆕 No hay data previa para Mes_Referencia = '202511'. Se insertará como archivo nuevo.
📥 Insertadas 18517 filas para Mes_Referencia = '202511'.

📊 Resumen de carga por Mes_Referencia:
   - 202511: 18517 filas cargadas (archivo nuevo).
